# COSC 661 · Combined HW3/4 — Colab Starter
Run with **GPU** (Runtime → Change runtime type). Upload `train.csv`, `test.csv`, and `evaluation.py` to `/content/`. This notebook downloads the model and requires internet on first run. Complete the TODO analysis in the homework report.

In [ ]:
!pip -q install "transformers>=4.45,<5" "peft>=0.13,<1" "accelerate>=1,<2" pandas

In [ ]:
import os, sys, time, random, numpy as np, pandas as pd, torch, transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model
from evaluation import format_compliant, summarize_outputs, validate_results, write_csv
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
print("torch",torch.__version__,"transformers",transformers.__version__,"peft",peft.__version__)
print("GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")
assert torch.cuda.is_available(), "This LoRA lab requires a GPU runtime."

In [ ]:
train_df=pd.read_csv("/content/train.csv"); test_df=pd.read_csv("/content/test.csv")
assert len(train_df)==160 and len(test_df)==20
assert not(set(train_df.id)&set(test_df.id))
print(train_df.head().to_string(index=False));print("Train/test:",len(train_df),len(test_df))

## Model loading
**Recommended:** `Qwen/Qwen2.5-0.5B-Instruct`. The model is downloaded from Hugging Face; no API key is required for public access. A Colab T4 or better is recommended.

In [ ]:
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to("cuda")
model.config.use_cache=False
print("Parameters:",sum(p.numel() for p in model.parameters()))

## Part A — prompts and decoding
Few-shot examples come only from **training** rows. Avoid including test references in prompts.

In [ ]:
examples=train_df.head(3).to_dict("records")
def basic_prompt(text):
    return f"Rewrite this sentence in a polite, professional tone. Return only the rewrite.\nSentence: {text}\nRewrite:"
def improved_prompt(text):
    demos="\n".join(f"Sentence: {r['input']}\nRewrite: {r['reference']}" for r in examples)
    return ("You are a workplace communication editor. Preserve meaning, add no new information, "
            "use a polite professional tone, and return only one rewritten sentence.\n"+demos+f"\nSentence: {text}\nRewrite:")
def generate(model, prompt, sample=False):
    messages=[{"role":"user","content":prompt}]
    rendered=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    batch=tokenizer(rendered,return_tensors="pt").to("cuda")
    args=dict(max_new_tokens=65,pad_token_id=tokenizer.eos_token_id,do_sample=sample)
    if sample: args.update(temperature=0.7,top_p=0.9)
    model.eval()
    with torch.no_grad(): out=model.generate(**batch,**args)
    return tokenizer.decode(out[0,batch["input_ids"].shape[1]:],skip_special_tokens=True).strip()

In [ ]:
decoding_rows=[]
for r in test_df.head(5).to_dict("records"):
    for name,sample in [("greedy",False),("top_p_sampling",True)]:
        decoding_rows.append({"id":r["id"],"strategy":name,"input":r["input"],"output":generate(model,improved_prompt(r["input"]),sample)})
pd.DataFrame(decoding_rows).to_csv("decoding_comparison.csv",index=False)
pd.DataFrame(decoding_rows).head(10)

## Part B — LoRA supervised adaptation
The loss is calculated **only on the target response**, not on the instruction prefix. This is important for valid supervised instruction tuning.

In [ ]:
MAX_LENGTH=192
class RewritingDataset(torch.utils.data.Dataset):
    def __init__(self,df):
        self.rows=df.to_dict("records")
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        prefix=tokenizer.apply_chat_template([{"role":"user","content":basic_prompt(r["input"])}],tokenize=False,add_generation_prompt=True)
        prefix_ids=tokenizer(prefix,add_special_tokens=False)["input_ids"]
        answer_ids=tokenizer(r["reference"]+tokenizer.eos_token,add_special_tokens=False)["input_ids"]
        ids=(prefix_ids+answer_ids)[:MAX_LENGTH]
        labels=([-100]*len(prefix_ids)+answer_ids)[:MAX_LENGTH]
        assert any(x!=-100 for x in labels), "Response truncated; increase max length"
        return {"input_ids":ids,"attention_mask":[1]*len(ids),"labels":labels}
def collate(features):
    max_len=max(len(f["input_ids"]) for f in features)
    batch={}
    for key in ("input_ids","attention_mask","labels"):
        pad_value=-100 if key=="labels" else (tokenizer.pad_token_id if key=="input_ids" else 0)
        batch[key]=torch.tensor([f[key]+[pad_value]*(max_len-len(f[key])) for f in features],dtype=torch.long)
    return batch
training_dataset=RewritingDataset(train_df)
print("First training example length",len(training_dataset[0]["input_ids"]))

In [ ]:
config=LoraConfig(task_type=TaskType.CAUSAL_LM,r=8,lora_alpha=16,lora_dropout=0.05,target_modules=["q_proj","v_proj"])
model=get_peft_model(model,config);model.print_trainable_parameters()
model.config.use_cache=False

In [ ]:
args=TrainingArguments(output_dir="/content/hw34_checkpoints",num_train_epochs=1,per_device_train_batch_size=2,gradient_accumulation_steps=4,learning_rate=2e-4,logging_steps=5,save_strategy="no",report_to="none",fp16=True,remove_unused_columns=False,seed=SEED)
trainer=Trainer(model=model,args=args,train_dataset=training_dataset,data_collator=collate)
t0=time.time();train_output=trainer.train();print("Training seconds",round(time.time()-t0,1),"train loss",train_output.training_loss)
model.save_pretrained("/content/hw34_lora_adapter");tokenizer.save_pretrained("/content/hw34_lora_adapter")

## Part C — Controlled evaluation
To avoid loading two model copies into GPU memory, unload the LoRA model after producing condition C, then reload the base model for A and B. All conditions use greedy decoding.

In [ ]:
results=[]
for r in test_df.to_dict("records"):
    results.append({"id":r["id"],"condition":"C","input":r["input"],"output":generate(model,basic_prompt(r["input"]))})
del trainer,model
import gc;gc.collect();torch.cuda.empty_cache()
base_model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16).to("cuda")
for r in test_df.to_dict("records"):
    for condition,prompt in [("A",basic_prompt(r["input"])),("B",improved_prompt(r["input"]))]:
        results.append({"id":r["id"],"condition":condition,"input":r["input"],"output":generate(base_model,prompt)})
validate_results(results,expected_ids=test_df.id.tolist())
write_csv("results.csv",results,["id","condition","input","output"])
print(summarize_outputs(results))

## Human rating sheet
Rate the **same 10 test IDs** across all three conditions, using 1–5 for meaning preservation, professional tone and fluency. Do not use the reference as the only valid wording.

In [ ]:
selected_ids=test_df.id.head(10).tolist()
ratings=[{"id":r["id"],"condition":r["condition"],"meaning_1to5":"","tone_1to5":"","fluency_1to5":"","notes":""} for r in results if r["id"] in selected_ids]
write_csv("human_ratings.csv",ratings,["id","condition","meaning_1to5","tone_1to5","fluency_1to5","notes"])
print("Wrote results.csv, decoding_comparison.csv, human_ratings.csv, and adapter directory")

## Report prompts
1. Explain how A, B, and C isolate prompt design versus adaptation.
2. Report format compliance for all 20 test items per condition.
3. After filling human_ratings.csv, report mean ratings per condition and three failure cases.
4. Discuss limitations of synthetic, template-like training data.
5. Explain why a single run cannot establish statistical significance or general performance.
6. Record model ID, GPU, package versions, and training parameters.

In [ ]:
from google.colab import files
# Download outputs after completing your manual scoring and report.
# files.download("results.csv")
# files.download("human_ratings.csv")